# 02 — Classification K-means

Le notebook agrège les métriques d'**un seul corpus**, applique la même préparation
numérique que l'ACP, compare plusieurs valeurs de `k`, puis ajuste le K-means final.
Les résultats sont écrits dans `analysis/<corpus>/kmeans/`.


## 1. Configuration


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from metric_registry import get_dataset
from pipeline_utils import detect_project_dir, aggregate_metrics, prepare_numeric_features

PROJECT_DIR = detect_project_dir()
DATASET = "Mirabelle"  # "Mirabelle", "Nowledgeable" ou "Progsnap2"
JOIN_MODE = "outer"
IMPUTATION = "median"  # "median", "mean" ou "drop_rows"
MIN_NON_NULL_FEATURES = 2
DROP_FEATURES = []

N_CLUSTERS = 4
K_RANGE = range(2, 9)
RANDOM_STATE = 42
N_INIT = 20
TOP_N_PROFILE_VARIABLES = 12

SPEC = get_dataset(DATASET)
CSV_DIR = SPEC.csv_dir(PROJECT_DIR)
ANALYSIS_DIR = SPEC.analysis_dir(PROJECT_DIR) / "kmeans"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print("Corpus  :", DATASET)
print("CSV     :", CSV_DIR)
print("Analyse :", ANALYSIS_DIR)


## 2. Agrégation et préparation


In [ ]:
features, inventory_df = aggregate_metrics(CSV_DIR, join_mode=JOIN_MODE)
display(inventory_df)

X_raw, usable_cols, removed_df = prepare_numeric_features(
    features,
    drop_features=DROP_FEATURES,
    min_non_null_features=MIN_NON_NULL_FEATURES,
)
if len(usable_cols) < 2:
    raise ValueError("Il faut au moins deux variables numériques non constantes pour K-means.")
if not removed_df.empty:
    display(removed_df)

subject_ids = X_raw["SubjectID"].astype(str).reset_index(drop=True)
X_values = X_raw[usable_cols].copy()

if IMPUTATION == "drop_rows":
    keep = X_values.notna().all(axis=1)
    X_values = X_values.loc[keep].reset_index(drop=True)
    subject_ids = subject_ids.loc[keep].reset_index(drop=True)
elif IMPUTATION in {"median", "mean"}:
    imputer = SimpleImputer(strategy=IMPUTATION)
    X_values = pd.DataFrame(imputer.fit_transform(X_values), columns=usable_cols)
else:
    raise ValueError("IMPUTATION doit valoir 'median', 'mean' ou 'drop_rows'.")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_values)

features_for_kmeans = pd.concat([subject_ids.rename("SubjectID"), X_values], axis=1)
features_for_kmeans.to_csv(ANALYSIS_DIR / "features_for_kmeans.csv", index=False)
print(f"{len(subject_ids)} étudiant(s), {len(usable_cols)} variable(s).")


## 3. Aide au choix de k


In [ ]:
rows = []
for k in [k for k in K_RANGE if 2 <= k <= len(subject_ids) - 1]:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT)
    labels = model.fit_predict(X_scaled)
    rows.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette": silhouette_score(X_scaled, labels),
    })
model_selection_df = pd.DataFrame(rows)
model_selection_df.to_csv(ANALYSIS_DIR / "kmeans_model_selection.csv", index=False)
display(model_selection_df)

if not model_selection_df.empty:
    plt.figure(figsize=(7, 4))
    plt.plot(model_selection_df["k"], model_selection_df["inertia"], marker="o")
    plt.xlabel("k")
    plt.ylabel("Inertie")
    plt.title(f"{DATASET} — méthode du coude")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(model_selection_df["k"], model_selection_df["silhouette"], marker="o")
    plt.xlabel("k")
    plt.ylabel("Score de silhouette")
    plt.title(f"{DATASET} — silhouette selon k")
    plt.tight_layout()
    plt.show()


## 4. Modèle final


In [ ]:
if not (2 <= N_CLUSTERS <= len(subject_ids)):
    raise ValueError(f"N_CLUSTERS doit être compris entre 2 et {len(subject_ids)}.")

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=N_INIT)
labels = kmeans.fit_predict(X_scaled)

clustered = features_for_kmeans.copy()
clustered["Cluster"] = labels
clustered.to_csv(ANALYSIS_DIR / "features_with_kmeans_clusters.csv", index=False)
clustered[["SubjectID", "Cluster"]].to_csv(ANALYSIS_DIR / "kmeans_clusters.csv", index=False)

centers_scaled_df = pd.DataFrame(kmeans.cluster_centers_, columns=usable_cols)
centers_scaled_df.index.name = "Cluster"
centers_scaled_df.to_csv(ANALYSIS_DIR / "kmeans_cluster_centers_scaled.csv")

centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
centers_original_df = pd.DataFrame(centers_original, columns=usable_cols)
centers_original_df.index.name = "Cluster"
centers_original_df.to_csv(ANALYSIS_DIR / "kmeans_cluster_centers_original_units.csv")

cluster_summary_df = clustered.groupby("Cluster").size().rename("n_sujets").reset_index()
cluster_summary_df.to_csv(ANALYSIS_DIR / "kmeans_cluster_summary.csv", index=False)
display(cluster_summary_df)


## 5. Profil des clusters


In [ ]:
profile_rows = []
for cluster_id, row in centers_scaled_df.iterrows():
    top = row.abs().sort_values(ascending=False).head(min(TOP_N_PROFILE_VARIABLES, len(row)))
    for variable in top.index:
        profile_rows.append({
            "Cluster": cluster_id,
            "variable": variable,
            "centre_standardise": row[variable],
            "direction": "au-dessus" if row[variable] > 0 else "au-dessous",
        })
profile_df = pd.DataFrame(profile_rows)
profile_df.to_csv(ANALYSIS_DIR / "kmeans_cluster_profile.csv", index=False)
display(profile_df)

# Visualise les variables qui séparent le plus fortement au moins un cluster.
discriminating = centers_scaled_df.abs().max(axis=0).sort_values(ascending=False)
cols_to_plot = discriminating.head(min(TOP_N_PROFILE_VARIABLES, len(discriminating))).index
plot_df = centers_scaled_df[cols_to_plot].T

plt.figure(figsize=(9, max(5, 0.38 * len(cols_to_plot))))
for cluster_id in plot_df.columns:
    plt.plot(plot_df.index, plot_df[cluster_id], marker="o", label=f"Cluster {cluster_id}")
plt.axhline(0, linewidth=0.8)
plt.xticks(rotation=90)
plt.ylabel("Centre standardisé")
plt.title(f"{DATASET} — profils des clusters")
plt.legend()
plt.tight_layout()
plt.show()
